# AME 5052 – Applied Machine Learning Lab
## Assignment 1: Regression and Classification Models

**Topics:** Linear Regression, Ridge, Lasso, Decision Tree, Logistic Regression, K-Nearest Neighbours (KNN), and Naive Bayes.

**Note:** This notebook is self-contained and uses datasets available through scikit-learn so it can run without downloading the GitHub repository. Replace the dataset-loading cells with your repository dataset paths if required by your instructor.

## 1(a). Linear Regression, Ridge and Lasso
We use the Diabetes dataset for a regression task. The target is a continuous disease-progression measure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

diabetes = load_diabetes(as_frame=True)
X_reg = diabetes.data
y_reg = diabetes.target

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

regression_models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge Regression": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso Regression": make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=10000))
}

regression_results = []
for name, model in regression_models.items():
    model.fit(X_train_reg, y_train_reg)
    pred = model.predict(X_test_reg)
    regression_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test_reg, pred),
        "MSE": mean_squared_error(y_test_reg, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test_reg, pred)),
        "R2 Score": r2_score(y_test_reg, pred)
    })

regression_results = pd.DataFrame(regression_results)
display(regression_results.round(3))

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test_reg, regression_models["Linear Regression"].predict(X_test_reg), alpha=0.7)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Linear Regression: Actual vs Predicted")
plt.grid(True)
plt.show()

**Analysis:** MAE, MSE and RMSE measure prediction error; lower values indicate better predictions. R² measures the proportion of target variance explained by the model; values closer to 1 indicate a better fit. Ridge uses L2 regularization, while Lasso uses L1 regularization and can shrink some coefficients to zero.

## 1(b). Decision Tree Classification
The Iris dataset is used to demonstrate classification and the Gini and entropy splitting criteria.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

iris = load_iris(as_frame=True)
X_tree = iris.data
y_tree = iris.target

X_train_tree, X_test_tree, y_train_tree, y_test_tree = train_test_split(
    X_tree, y_tree, test_size=0.2, random_state=42, stratify=y_tree
)

tree_models = {
    "Gini": DecisionTreeClassifier(criterion="gini", random_state=42, max_depth=3),
    "Entropy": DecisionTreeClassifier(criterion="entropy", random_state=42, max_depth=3)
}

for name, model in tree_models.items():
    model.fit(X_train_tree, y_train_tree)
    predictions = model.predict(X_test_tree)
    print(f"Decision Tree using {name}")
    print("Accuracy:", round(accuracy_score(y_test_tree, predictions), 4))
    print(classification_report(y_test_tree, predictions, target_names=iris.target_names))
    print("Confusion matrix:\n", confusion_matrix(y_test_tree, predictions))

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(tree_models["Entropy"], feature_names=iris.feature_names,
          class_names=iris.target_names, filled=True, rounded=True)
plt.title("Decision Tree using Entropy")
plt.show()

**Concepts:** Entropy measures impurity. Information gain is the reduction in entropy after a split. Gini impurity measures the probability of incorrectly classifying a randomly selected sample. The decision tree selects splits that reduce impurity.

## 1(c). Logistic Regression, Sigmoid Function, Decision Boundary, Ridge and Lasso
A binary classification version of the Breast Cancer Wisconsin dataset is used. Logistic Regression estimates probabilities using the sigmoid function.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

cancer = load_breast_cancer(as_frame=True)
X_log = cancer.data
y_log = cancer.target

X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_log, y_log, test_size=0.2, random_state=42, stratify=y_log
)

logistic_models = {
    "Logistic Regression (L2/Ridge)": make_pipeline(
        StandardScaler(), LogisticRegression(penalty="l2", C=1.0, max_iter=5000)
    ),
    "Logistic Regression (L1/Lasso)": make_pipeline(
        StandardScaler(), LogisticRegression(penalty="l1", solver="liblinear", C=1.0, max_iter=5000)
    )
}

for name, model in logistic_models.items():
    model.fit(X_train_log, y_train_log)
    pred = model.predict(X_test_log)
    print(name)
    print("Accuracy:", round(accuracy_score(y_test_log, pred), 4))
    print("Precision:", round(precision_score(y_test_log, pred), 4))
    print("Recall:", round(recall_score(y_test_log, pred), 4))
    print("F1-score:", round(f1_score(y_test_log, pred), 4))
    print("Confusion matrix:\n", confusion_matrix(y_test_log, pred))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z_values = np.linspace(-8, 8, 200)
plt.figure(figsize=(7, 4))
plt.plot(z_values, sigmoid(z_values))
plt.xlabel("z")
plt.ylabel("Sigmoid(z)")
plt.title("Sigmoid Function")
plt.grid(True)
plt.show()

**Explanation:** The sigmoid function maps any real-valued input to a probability between 0 and 1. A default threshold of 0.5 converts probability into a class prediction. L2 (Ridge) regularization penalizes squared coefficients; L1 (Lasso) regularization penalizes absolute coefficients and can produce sparse models.

## 1(d). K-Nearest Neighbours (KNN)
KNN classifies a sample based on the majority class among its nearest neighbours.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5, metric="minkowski", p=2)
)
knn_model.fit(X_train_tree, y_train_tree)
knn_pred = knn_model.predict(X_test_tree)

print("KNN Accuracy:", round(accuracy_score(y_test_tree, knn_pred), 4))
print(classification_report(y_test_tree, knn_pred, target_names=iris.target_names))
print("Confusion matrix:\n", confusion_matrix(y_test_tree, knn_pred))

**Analysis:** KNN is a distance-based, non-parametric algorithm. Feature scaling is important because features with larger numerical ranges can dominate distance calculations. The value of K affects the bias-variance trade-off.

## 1(e). Naive Bayes Classification
Gaussian Naive Bayes is applied to the Iris dataset.

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_tree, y_train_tree)
nb_pred = nb_model.predict(X_test_tree)

print("Naive Bayes Accuracy:", round(accuracy_score(y_test_tree, nb_pred), 4))
print(classification_report(y_test_tree, nb_pred, target_names=iris.target_names))
print("Confusion matrix:\n", confusion_matrix(y_test_tree, nb_pred))

**Conclusion:** The notebook implements regression and classification algorithms and reports suitable evaluation metrics. For regression, MAE, MSE, RMSE and R² are used. For classification, accuracy, precision, recall, F1-score and confusion matrix are used. The best model should be selected based on the metric that matches the practical problem.